# Major Project: Seasonal Agriculture Performance Analysis

**VOIS AICTE Batch 1, 2026–2027**

This notebook analyzes the supplied agricultural dataset to identify seasonal patterns, differences, relationships, and evidence-based recommendations.

## Project objective
- Explore and clean the dataset.
- Compare agricultural performance across Kharif, Rabi and Zaid seasons.
- Examine environmental conditions, resource usage and economic outcomes.
- Investigate crop and irrigation differences.
- Use statistical tests and visualizations to support conclusions.
- Document limitations and avoid interpreting correlation as causation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_csv("seasonal_agriculture_performance_dataset (3)(1).csv")
df.head()


## 1. Dataset structure and data quality

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum()[df.isna().sum() > 0].to_frame("missing"))

print("\nDuplicate rows:", df.duplicated().sum())


### Data-quality finding
The dataset contains 4,000 farm records and 28 columns. There are no duplicate rows. Missing values occur only in `Rainfall_mm`, `Soil_Moisture_pct`, and `Yield_Tonnes_Ha`, so analysis of each metric uses available observations. For descriptive visualization where a complete matrix is needed, median imputation may be used and must be clearly labeled.

In [ ]:
df.describe(include="all").T


## 2. Seasonal descriptive analysis

In [ ]:
season_order = ["Kharif", "Rabi", "Zaid"]
df["Season"] = pd.Categorical(df["Season"], categories=season_order, ordered=True)

season_kpi = df.groupby("Season", observed=True).agg(
    Farms=("Farm_ID","count"),
    Avg_Yield=("Yield_Tonnes_Ha","mean"),
    Median_Yield=("Yield_Tonnes_Ha","median"),
    Avg_Revenue=("Revenue_INR","mean"),
    Avg_Cost=("Total_Cost_INR","mean"),
    Avg_Profit=("Profit_INR","mean"),
    Profit_Positive_pct=("Profit_INR", lambda x: (x > 0).mean()*100),
    Avg_Water=("Water_Used_m3","mean"),
    Avg_Water_Eff=("Water_Efficiency_t_per_1000m3","mean"),
    Avg_Rainfall=("Rainfall_mm","mean"),
    Avg_Temp=("Avg_Temperature_C","mean"),
    Avg_Risk=("Disease_Pest_Risk_pct","mean")
)
for s in season_order:
    idx = df["Season"] == s
    season_kpi.loc[s, "Profit_Margin_pct"] = df.loc[idx,"Profit_INR"].sum()/df.loc[idx,"Revenue_INR"].sum()*100
display(season_kpi.round(2))


### Key interpretation
Kharif has the strongest overall average profit and the highest average rainfall and disease/pest risk. Zaid has the weakest average profitability and the lowest average rainfall. These are descriptive seasonal differences, not proof that season alone causes the outcomes.

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
x = np.arange(len(season_order))
ax.bar(x-0.2, season_kpi["Avg_Yield"], 0.4, label="Average yield (t/ha)")
ax.set_xticks(x, season_order)
ax.set_ylabel("Yield (t/ha)")
ax.set_title("Average Yield by Season")
ax.legend()
plt.show()


## 3. Crop composition and crop-by-season comparison

In [ ]:
crop_mix = pd.crosstab(df["Season"], df["Crop"], normalize="index") * 100
display(crop_mix.round(1))

crop_season_yield = df.pivot_table(
    index="Crop", columns="Season", values="Yield_Tonnes_Ha",
    aggfunc="mean", observed=False
).reindex(columns=season_order)
display(crop_season_yield.round(2))


A major analytical caution is **crop mix**. The dataset contains eight crops, and their yields differ substantially. Therefore, a raw seasonal average can partly reflect which crops are represented in each season. The crop-by-season table is a better way to check whether seasonal patterns remain visible within individual crops.

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
im = ax.imshow(crop_season_yield.values, aspect="auto")
ax.set_xticks(range(3), season_order)
ax.set_yticks(range(len(crop_season_yield.index)), crop_season_yield.index)
ax.set_title("Crop-wise Seasonal Yield")
plt.colorbar(im, ax=ax, label="Yield (t/ha)")
plt.show()


## 4. Statistical testing

In [ ]:
# Kruskal-Wallis tests are used because the distributions are strongly skewed,
# especially for yield and profit.
results = []
for metric in ["Yield_Tonnes_Ha","Profit_INR","Revenue_INR","Water_Used_m3","Disease_Pest_Risk_pct"]:
    groups = [g[metric].dropna() for _, g in df.groupby("Season", observed=True)]
    h, p = stats.kruskal(*groups)
    results.append([metric, h, p])
display(pd.DataFrame(results, columns=["Metric","Kruskal_H","p_value"]))

anova = stats.f_oneway(*[
    g["Yield_Tonnes_Ha"].dropna() for _, g in df.groupby("Season", observed=True)
])
print("One-way ANOVA for yield:", anova)


### Statistical interpretation
The Kruskal–Wallis test detects significant distribution differences across seasons for yield, profit, revenue and disease/pest risk at conventional significance levels, while the water-use result is borderline. Because yield is highly skewed by crop, the non-parametric result should be interpreted together with crop-level comparisons. A one-way ANOVA on raw yield is not significant in this dataset, illustrating why the choice of statistical method and distribution shape matter.

## 5. Irrigation analysis

In [ ]:
irr_stats = df.groupby("Irrigation_Method").agg(
    Farms=("Farm_ID","count"),
    Avg_Yield=("Yield_Tonnes_Ha","mean"),
    Avg_Profit=("Profit_INR","mean"),
    Avg_Water=("Water_Used_m3","mean"),
    Avg_Water_Eff=("Water_Efficiency_t_per_1000m3","mean")
).sort_values("Avg_Profit", ascending=False)
display(irr_stats.round(2))


In the supplied data, drip irrigation has the highest average profit and yield among irrigation methods. Rainfed farming has the lowest average water use and the highest calculated water-efficiency metric, but that metric is influenced by the formula used in the dataset and should not be treated as proof that rainfed systems are agronomically superior. Irrigation method is observational here, so causal claims require controlled or longitudinal evidence.

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
ax.bar(irr_stats.index, irr_stats["Avg_Profit"]/100000)
ax.set_ylabel("Average profit (₹ lakh)")
ax.set_title("Average Profit by Irrigation Method")
plt.xticks(rotation=15)
plt.show()


## 6. Environmental and risk patterns

In [ ]:
env = df.groupby("Season", observed=True).agg(
    Rainfall_mm=("Rainfall_mm","mean"),
    Temperature_C=("Avg_Temperature_C","mean"),
    Humidity_pct=("Humidity_pct","mean"),
    Sunlight_h=("Sunlight_Hours_Day","mean"),
    Soil_Moisture_pct=("Soil_Moisture_pct","mean"),
    Disease_Risk_pct=("Disease_Pest_Risk_pct","mean")
)
display(env.round(2))


## 7. Correlation analysis

In [ ]:
selected = [
    "Rainfall_mm","Avg_Temperature_C","Soil_Moisture_pct",
    "Fertilizer_kg_ha","Seed_Quality_Score","Water_Used_m3",
    "Disease_Pest_Risk_pct","Yield_Tonnes_Ha","Profit_INR"
]
corr = df[selected].corr()
display(corr.round(2))

fig, ax = plt.subplots(figsize=(8,6))
im = ax.imshow(corr.values, vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(selected)), selected, rotation=45, ha="right")
ax.set_yticks(range(len(selected)), selected)
ax.set_title("Correlation Matrix")
plt.colorbar(im, ax=ax, label="Pearson correlation")
plt.show()


### Correlation caution
Profit is positively correlated with yield and revenue in this dataset, but correlation does not establish causality. `Water_Efficiency_t_per_1000m3` is mathematically related to yield and water use, so a high correlation with yield is expected and should not be presented as an independent discovery.

## 8. Internal consistency checks

In [ ]:
prod_ratio = df["Production_Tonnes"] / (df["Farm_Area_Hectares"] * df["Yield_Tonnes_Ha"])
revenue_ratio = df["Revenue_INR"] / (df["Production_Tonnes"] * df["Market_Price_INR_Tonne"])
profit_diff = (df["Profit_INR"] - (df["Revenue_INR"] - df["Total_Cost_INR"])).abs()

print("Production / (area × yield):")
print(prod_ratio.describe())
print("\nRevenue / (production × market price):")
print(revenue_ratio.describe())
print("\nMaximum absolute profit reconciliation error:", profit_diff.max())


The core arithmetic relationships in the supplied data are internally consistent to rounding precision: production is approximately area × yield, revenue is approximately production × market price, and profit exactly reconciles to revenue − cost. This is useful for validating downstream analysis.

## 9. Conclusions and evidence-based recommendations

### Conclusions
1. **Kharif shows the strongest overall profitability** in the dataset, while Zaid shows the weakest.
2. **Seasonal yield differences are present in the distributions**, but raw averages are strongly affected by crop-specific yield differences, especially high-yield crops.
3. **Drip irrigation has the highest average profit and yield** among the irrigation categories in this observational dataset.
4. **Disease/pest risk varies strongly by season**, with the highest average risk in Kharif.
5. **Water use is not significantly different at the 5% level by Kruskal–Wallis testing**, although the descriptive averages differ.
6. The dataset contains a small amount of missing data and substantial skew/outliers, so median-based and non-parametric analysis is appropriate.
7. Strong relationships involving production, revenue, profit and water-efficiency should be interpreted carefully because some are mathematical relationships rather than independent causal effects.

### Recommendations
- Compare farms **within the same crop** before making seasonal planning decisions.
- Prioritize investigation of **drip-irrigated farms** as a potentially promising operational pattern, while avoiding causal claims without controlled evidence.
- Strengthen **Kharif pest/disease preparedness** because its observed risk level is highest.
- Treat Zaid as a higher-risk profitability period in this dataset and investigate its crop-specific cost/revenue structure.
- Use the notebook as a monitoring framework and add multi-year or farm-level longitudinal data for stronger causal analysis.
